# Notebook 22 – Computer Vision Mini Assessment



In [7]:
import numpy as np
import torch
from torch import nn
torch.manual_seed(42)
print('PyTorch:',torch.__version__)

PyTorch: 2.14.0+cpu


## 1. Conceptual: CNN building blocks
**Question:** What does a convolution filter learn? What is a feature map? Why do CNNs commonly use pooling, padding, and stride?

**Answer:** A filter learns weights that respond to patterns such as edges or textures. Applying it across an input produces a feature map of response strengths and locations. Pooling reduces spatial size; padding controls border handling and output dimensions; stride controls how far the filter moves each step.

## 2. Numerical: convolution and output shape
**Question A:** Apply kernel `[[1, 0], [-1, 1]]` with stride 1 and no padding to image `[[1, 2, 0], [0, 1, 3], [2, 1, 0]]`. What is the output?

**Answer A:** `[[2, 4], [-1, 0]]` using cross-correlation (the operation commonly called convolution in CNN libraries).

**Question B:** For input size 32, kernel 3, stride 2, and padding 1, compute output size with $\lfloor (N+2P-K)/S floor+1$.

**Answer B:** $\lfloor(32+2-3)/2floor+1=16$.

In [2]:
image=torch.tensor([[1,2,0],[0,1,3],[2,1,0]],dtype=torch.float32)[None,None]
kernel=torch.tensor([[1,0],[-1,1]],dtype=torch.float32)[None,None]
result=nn.functional.conv2d(image,kernel,stride=1,padding=0)
expected=torch.tensor([[2.,4.],[-1.,0.]])
assert torch.equal(result[0,0],expected)
output_size=(32+2*1-3)//2+1
assert output_size==16
print('convolution output:',result[0,0].int().tolist(),'| output size:',output_size)

convolution output: [[2, 4], [-1, 0]] | output size: 16


## 3. Comparison: padding and stride
**Question:** Compare `padding=0` with `padding=1` for a 3×3 filter, and compare stride 1 with stride 2. What is one trade-off of each?

**Answer:** No padding shrinks the spatial dimensions and discards border context; padding 1 preserves dimensions at stride 1. Stride 1 keeps more spatial detail but costs more compute; stride 2 reduces output size and computation but can miss fine details.

## 4. Scenario: augmentation and label validity
**Question:** A medical image classifier predicts left-versus-right pathology. Should horizontal flips be used automatically? Name a safer augmentation principle.

**Answer:** No. A horizontal flip can swap laterality and invalidate the label. Use only transformations that preserve the target meaning, and confirm with domain knowledge. Small rotations or brightness changes may be suitable only if they reflect realistic acquisition variation.

## 5. Conceptual/debugging: BatchNorm and Dropout
**Question A:** What does BatchNorm do, and how does it differ from standardizing input images? What does Dropout do?

**Answer A:** Input standardization transforms data before the model. BatchNorm normalizes intermediate activations using batch statistics during training and running estimates during evaluation, with learned scale and shift. Dropout randomly zeros activations during training to regularize.

**Question B:** A model is accidentally left in `eval()` while training. What can go wrong?

**Answer B:** Dropout stays disabled and BatchNorm running statistics are not updated. Call `model.train()` for training and `model.eval()` for evaluation; use `torch.no_grad()` during inference when gradients are unnecessary.

## 6. Comparison: transfer learning and model families
**Question:** Explain feature extraction versus fine-tuning. Compare the design emphasis of ResNet, MobileNet, and EfficientNet.

**Answer:** Feature extraction freezes a pretrained backbone and trains a new head. Fine-tuning unfreezes selected backbone layers and adapts them with a small learning rate. ResNet uses residual skip connections to train deeper networks; MobileNet uses efficient depthwise-separable convolutions for constrained devices; EfficientNet balances depth, width, and input resolution through compound scaling. The best choice depends on accuracy, hardware, latency, and data.

## 7. Scenario: transfer learning
**Question:** You have 400 labeled images from a domain somewhat related to ImageNet. What training order would you try, and what is the risk of unfreezing every layer immediately?

**Answer:** Load a pretrained model, replace its classifier, freeze the backbone, train the head, then unfreeze late layers and fine-tune with a smaller learning rate if validation supports it. Unfreezing everything immediately can overfit the small dataset and destroy useful pretrained features through large updates.

## 8. Numerical: IoU and object detection
**Question A:** Detection boxes are `A=[0,0,4,4]` and `B=[2,1,6,4]` in `(x_min,y_min,x_max,y_max)` format. Compute IoU.

**Answer A:** Areas are 16 and 12; intersection is $2×3=6$; union is $16+12-6=22$; IoU is $6/22≈0.273$.

**Question B:** What do confidence scoring and Non-Maximum Suppression (NMS) do?

**Answer B:** A detector predicts classes, boxes, and confidence scores. NMS keeps high-scoring boxes and suppresses overlapping lower-scoring boxes for the same class, using an IoU threshold. NMS removes duplicate predictions; it does not improve the detector's underlying localization.

In [3]:
def box_iou(box_a,box_b):
    ax1,ay1,ax2,ay2=box_a; bx1,by1,bx2,by2=box_b
    iw=max(0,min(ax2,bx2)-max(ax1,bx1)); ih=max(0,min(ay2,by2)-max(ay1,by1))
    intersection=iw*ih
    area_a=max(0,ax2-ax1)*max(0,ay2-ay1); area_b=max(0,bx2-bx1)*max(0,by2-by1)
    union=area_a+area_b-intersection
    return intersection/union if union else 0.0

def nms(boxes,scores,iou_threshold=.5):
    order=np.argsort(scores)[::-1]; keep=[]
    while len(order):
        current=order[0]; keep.append(int(current))
        remaining=[idx for idx in order[1:] if box_iou(boxes[current],boxes[idx])<=iou_threshold]
        order=np.asarray(remaining,dtype=int)
    return keep
boxes=np.array([[0,0,4,4],[.2,.2,4.1,4.1],[8,8,10,10]],dtype=float)
scores=np.array([.9,.8,.7])
assert abs(box_iou(boxes[0],boxes[1])-0.8610614192)<1e-6
kept=nms(boxes,scores,.5)
assert kept==[0,2]
print('IoU:',round(box_iou([0,0,4,4],[2,1,6,4]),3),'| NMS kept indices:',kept)

IoU: 0.273 | NMS kept indices: [0, 2]


## 9. Conceptual: image segmentation
**Question:** Distinguish semantic, instance, and panoptic segmentation. When are masks more useful than boxes?

**Answer:** Semantic segmentation assigns a category to each pixel but does not separate same-class objects. Instance segmentation gives each object a separate mask and class. Panoptic segmentation combines semantic labels for background/stuff with distinct instances for countable objects. Masks are useful when exact shape, boundaries, or pixel area matter; boxes are cheaper and sufficient for coarse localization.

In [4]:
def binary_iou(prediction,target):
    prediction=np.asarray(prediction,dtype=bool); target=np.asarray(target,dtype=bool)
    union=np.logical_or(prediction,target).sum()
    return np.logical_and(prediction,target).sum()/union if union else 1.0
predicted_mask=np.array([[1,1,0],[0,1,0]],dtype=bool)
target_mask=np.array([[1,0,0],[0,1,1]],dtype=bool)
score=binary_iou(predicted_mask,target_mask)
assert abs(score-0.5)<1e-8
print('mask IoU:',score)

mask IoU: 0.5


## 10. Comparison: deployment optimization
**Question:** A large CNN is 1.5 points more accurate but 8× larger and 4× slower than a small model. Which should ship?

**Answer:** There is no universal winner. Choose based on minimum accuracy, latency, memory, energy, and hardware requirements. If the small model meets quality thresholds and the larger model violates latency or memory budgets, the smaller model is more practical. Benchmark batch and single-image inference on target hardware; quantization/ONNX may help but must be measured for quality and speed.

## 11. Scenario: error analysis
**Question:** A classifier has 93% accuracy, but nearly all errors are in one rare class and those errors have high business cost. What should you inspect and report?

**Answer:** Inspect class support, per-class precision/recall/F1, confusion matrix, confidence, and representative errors. Check label quality, image conditions, and leakage/split design. Consider class-weighted loss, balanced sampling, targeted data collection, or valid augmentation. Compare on a fixed validation set and report the rare-class metric and cost, not accuracy alone.

## 12. Coding: construct a CNN and verify output dimensions
**Question:** Build a small CNN for 1-channel 28×28 images and 10 classes using convolution, BatchNorm, ReLU, pooling, Dropout, and a fully connected output. Verify logits have shape `(batch, 10)`.

**Answer: run the code cell below.**

In [5]:
class AssessmentCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features=nn.Sequential(
            nn.Conv2d(1,16,kernel_size=3,padding=1),
            nn.BatchNorm2d(16),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(16,32,kernel_size=3,stride=1,padding=1),
            nn.ReLU(),nn.MaxPool2d(2),
        )
        self.classifier=nn.Sequential(nn.Flatten(),nn.Dropout(.25),nn.Linear(32*7*7,10))
    def forward(self,image): return self.classifier(self.features(image))
model=AssessmentCNN()
logits=model(torch.randn(4,1,28,28))
assert logits.shape==(4,10)
print('CNN output shape:',tuple(logits.shape))

CNN output shape: (4, 10)


## 13. Debugging: training versus evaluation mode
**Question:** Find and fix the bug: `model.eval(); for images,labels in train_loader: loss=model(images).sum(); loss.backward()`.

**Answer:** The model is in evaluation mode and no optimizer update is shown. For training, call `model.train()`, zero optimizer gradients, compute a task loss, call `backward()`, then `optimizer.step()`. Use `model.eval()` and `torch.no_grad()` only for evaluation/inference.

In [6]:
debug_model=AssessmentCNN()
optimizer=torch.optim.SGD(debug_model.parameters(),lr=.01)
images=torch.randn(8,1,28,28); labels=torch.randint(0,10,(8,))
debug_model.train(); optimizer.zero_grad()
loss=nn.functional.cross_entropy(debug_model(images),labels)
loss.backward(); optimizer.step()
debug_model.eval()
with torch.no_grad(): evaluation_logits=debug_model(images)
assert evaluation_logits.shape==(8,10)
print('training loss:',round(loss.item(),3),'| eval shape:',tuple(evaluation_logits.shape))

training loss: 2.189 | eval shape: (8, 10)


## 14. Coding: identify a shape/debugging issue
**Question:** A 3×3 convolution receives `(N, 3, 32, 32)` input but is declared `nn.Conv2d(1, 16, 3)`. What is wrong, and how do you fix it?

**Answer:** The convolution expects one input channel but receives three RGB channels. Declare `nn.Conv2d(3,16,3)` or explicitly convert the image to grayscale. Keep channel count and preprocessing consistent throughout the pipeline.

## 15. Short answer: compare and justify
**Question:** Choose between a ResNet, MobileNet, and EfficientNet for (a) a phone, (b) a high-accuracy server, and (c) a transfer-learning baseline. Explain what you would measure.

**Answer:** MobileNet is a natural phone candidate because it targets efficient inference. EfficientNet may offer a strong accuracy/compute balance for a server, subject to benchmarking. ResNet is a robust, widely supported transfer-learning baseline. Measure validation quality, class-specific errors, batch and single-image latency, memory, model size, and target-device compatibility; architecture names alone do not decide.